Description: Get peak information, perform coincidence, and plot summed spectrum

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import sys
import glob
import os
sys.path.insert(0,"/home/ws/sk6801/sw/UCSD_analysis/sandpro")
import sandpro
import configparser
import json
import scipy.stats
from matplotlib.colors import LogNorm

from scipy.optimize import curve_fit
import datetime
import pandas as pd
from copy import deepcopy
from numba import jit
import time

sys.path.insert(0,"../src/")
import common.d2d as d2d
import common.utils as util
import data_processing.fast_processor_all_channel as fast_processor
from data_processing.event_processor_all_channel import EventProcessor

from data_structure.waveform_info import WaveformInfo
from data_structure.peak_info import PeakInfo

%run "plot_style_kalinka.py"


#### Load data
It was observed that there is no significant difference between board 0 and 1 and all -> go to board 0 channel 0 only

In [ ]:
df_event_rate = pd.read_csv(
    "/kalinka/storage/darkmatter/XENONnT/sk6801/UCSD_data/processed_data/peak_all_channel_event_rate_20250812_133321.csv",
    parse_dates=["absolute_time"]
    )

In [ ]:
df_important = pd.read_csv(
    "/kalinka/storage/darkmatter/XENONnT/sk6801/UCSD_data/processed_data/important_time_stamp.csv", 
    parse_dates=["date_time"]
    )

#### Plots

In [ ]:
plt.hist(df_event_rate.integral_area_Vns_board_channel0, bins = 100)

plt.yscale('log')

##### Binned in time for the tritium data

In [ ]:
fig, ax = plt.subplots(figsize=(20, 4))


mask = (df_event_rate.absolute_time >= pd.Timestamp("2024-10-30")) & (
    df_event_rate.board == 0) & (
        df_event_rate.integral_area_Vns_board_channel0 > 0)
ax.hist(df_event_rate.absolute_time[mask], bins=1500)

for date_time, comment in zip(df_important.date_time, df_important.comment):
    ax.axvline(date_time, color='r', linestyle="dashed")
    ax.text(date_time, 0.1, 
            


            
            f"{comment}", 
            rotation=90, 
            ha='left', 
            fontsize=12, 
            color='black')

# rotate x axis
plt.xticks(rotation=45)


##### Across different bias voltage

In [ ]:
mask = (df_event_rate.absolute_time >= pd.Timestamp("2024-10-30 00:00:00")) & (
    df_event_rate.board == 0) & (
        df_event_rate.integral_area_PE_board_channel0 > 0)
tritium_data = df_event_rate[mask]

number_of_events = tritium_data.groupby(['md_full_path'])['board'].count().to_numpy()
start_time = tritium_data.groupby(['md_full_path'])['absolute_time'].min()
end_time = tritium_data.groupby(['md_full_path'])['absolute_time'].max()
runtime_s = tritium_data.groupby(['md_full_path'])['runtime_s'].mean().to_numpy()
area = tritium_data.groupby(['md_full_path'])['integral_area_PE_board_channel0'].mean().to_numpy()
threshold_adc = tritium_data.groupby(['md_full_path'])['threshold_adc'].mean().to_numpy()
voltage_preamp1_V = tritium_data.groupby(['md_full_path'])['voltage_preamp1_V'].mean().to_numpy()


delta_time = (end_time - start_time).dt.total_seconds()
event_rate = number_of_events / delta_time

fig, ax = plt.subplots(figsize=(20, 4))

for voltage in np.unique(voltage_preamp1_V):
    mask = voltage_preamp1_V == voltage
    ax.scatter(start_time[mask], event_rate[mask], s = area[mask], 
               label=f"Voltage: {voltage:.2f} V")
    
    # ax.scatter(start_time[mask], threshold_adc[mask], s = area[mask], color='green')
# ax.plot(start_time, event_rate, '-')


for date_time, comment in zip(df_important.date_time, df_important.comment):
    ax.axvline(date_time, color='r', linestyle="dashed")
    ax.text(date_time, 0.1, 
            f"{comment}", 
            rotation=90, 
            ha='left', 
            fontsize=12, 
            color='black')

# rotate x axis
plt.xticks(rotation=45)

plt.legend(bbox_to_anchor=(1.05, 0.5), loc='center left', fontsize=12)


##### Playing with different cuts etc

In [ ]:
mask = (df_event_rate.absolute_time >= pd.Timestamp("2024-10-30 00:00:00")) & (
        util.vec_regex_search("10.0sig", df_event_rate.md_full_path)) & (
        df_event_rate.voltage_preamp1_V == -46) & (
        df_event_rate.board == 0) & (
        df_event_rate.integral_area_PE_board_channel0 > 0)

tritium_data = df_event_rate[mask]

number_of_events = tritium_data.groupby(['md_full_path'])['board'].count().to_numpy()
start_time = tritium_data.groupby(['md_full_path'])['absolute_time'].min()
end_time = tritium_data.groupby(['md_full_path'])['absolute_time'].max()
area = tritium_data.groupby(['md_full_path'])['integral_area_PE_board_channel0'].mean().to_numpy()
threshold_adc = tritium_data.groupby(['md_full_path'])['threshold_adc'].mean().to_numpy()
voltage_preamp1_V = tritium_data.groupby(['md_full_path'])['voltage_preamp1_V'].mean().to_numpy()


delta_time = (end_time - start_time).dt.total_seconds()
event_rate = number_of_events / delta_time

fig, ax = plt.subplots(figsize=(20, 4))

for voltage in np.unique(voltage_preamp1_V):
    mask = voltage_preamp1_V == voltage
    ax.scatter(start_time[mask], event_rate[mask], s = area[mask], 
               label=f"Voltage: {voltage:.2f} V")
    

for date_time, comment in zip(df_important.date_time, df_important.comment):
    ax.axvline(date_time, color='r', linestyle="dashed")
    ax.text(date_time, 0.1, 
            f"{comment}", 
            rotation=90, 
            ha='left', 
            fontsize=12, 
            color='black')

# rotate x axis
plt.xticks(rotation=45)

plt.legend(bbox_to_anchor=(1.05, 0.5), loc='center left', fontsize=12)


In [ ]:
mask = (df_event_rate.absolute_time >= pd.Timestamp("2024-10-30 00:00:00")) & (
        util.vec_regex_search("10.0sig", df_event_rate.md_full_path)) & (
        df_event_rate.voltage_preamp1_V == -46) & (
        df_event_rate.board == 0) & (
        df_event_rate.integral_area_PE_board_channel0 > 10)

tritium_data = df_event_rate[mask]

number_of_events = tritium_data.groupby(['md_full_path'])['board'].count().to_numpy()
start_time = tritium_data.groupby(['md_full_path'])['absolute_time'].min()
end_time = tritium_data.groupby(['md_full_path'])['absolute_time'].max()
area = tritium_data.groupby(['md_full_path'])['integral_area_PE_board_channel0'].mean().to_numpy()
threshold_adc = tritium_data.groupby(['md_full_path'])['threshold_adc'].mean().to_numpy()
voltage_preamp1_V = tritium_data.groupby(['md_full_path'])['voltage_preamp1_V'].mean().to_numpy()
runtime_s = tritium_data.groupby(['md_full_path'])['runtime_s'].mean().to_numpy()


delta_time = (end_time - start_time).dt.total_seconds()
event_rate = number_of_events / delta_time

fig, ax = plt.subplots(figsize=(20, 4))

for voltage in np.unique(voltage_preamp1_V):
    mask = voltage_preamp1_V == voltage
    ax.scatter(start_time[mask], event_rate[mask], s = area[mask], 
               label=f"Voltage: {voltage:.2f} V")
    

for date_time, comment in zip(df_important.date_time, df_important.comment):
    ax.axvline(date_time, color='r', linestyle="dashed")
    ax.text(date_time, 0.1, 
            f"{comment}", 
            rotation=90, 
            ha='left', 
            fontsize=12, 
            color='black')

# rotate x axis
plt.xticks(rotation=45)

plt.legend(bbox_to_anchor=(1.05, 0.5), loc='center left', fontsize=12)


In [ ]:
mask = (df_event_rate.absolute_time >= pd.Timestamp("2024-10-30 00:00:00")) & (
        util.vec_regex_search("10.0sig", df_event_rate.md_full_path)) & (
        df_event_rate.voltage_preamp1_V == -46) & (
        df_event_rate.board == 0) & (
        df_event_rate.integral_area_PE_board_channel0 > 10)

tritium_data = df_event_rate[mask]

number_of_events = tritium_data.groupby(['md_full_path'])['board'].count().to_numpy()
start_time = tritium_data.groupby(['md_full_path'])['absolute_time'].min()
end_time = tritium_data.groupby(['md_full_path'])['absolute_time'].max()
area = tritium_data.groupby(['md_full_path'])['integral_area_PE_board_channel0'].mean().to_numpy()
threshold_adc = tritium_data.groupby(['md_full_path'])['threshold_adc'].mean().to_numpy()
voltage_preamp1_V = tritium_data.groupby(['md_full_path'])['voltage_preamp1_V'].mean().to_numpy()
runtime_s = tritium_data.groupby(['md_full_path'])['runtime_s'].mean().to_numpy()


delta_time = (end_time - start_time).dt.total_seconds()
event_rate = number_of_events / runtime_s

fig, ax = plt.subplots(figsize=(20, 4))

for voltage in np.unique(voltage_preamp1_V):
    mask = voltage_preamp1_V == voltage
    ax.scatter(start_time[mask], event_rate[mask], s = area[mask], 
               label=f"Voltage: {voltage:.2f} V")
    

for date_time, comment in zip(df_important.date_time, df_important.comment):
    ax.axvline(date_time, color='r', linestyle="dashed")
    ax.text(date_time, 0.1, 
            f"{comment}", 
            rotation=90, 
            ha='left', 
            fontsize=12, 
            color='black')

# rotate x axis
plt.xticks(rotation=45)

plt.legend(bbox_to_anchor=(1.05, 0.5), loc='center left', fontsize=12)


In [ ]:



mask = (df_event_rate.absolute_time >= pd.Timestamp("2024-10-30 00:00:00")) & (
        util.vec_regex_search("10.0sig", df_event_rate.md_full_path)) & (
        df_event_rate.voltage_preamp1_V == -46) & (
        df_event_rate.board == 0) & (
        df_event_rate.integral_area_PE_board_channel0 > 10)

tritium_data = df_event_rate[mask]
start_time = tritium_data['absolute_time'].min()
end_time = tritium_data['absolute_time'].max()
nbins = int((end_time - start_time).total_seconds())


# convert time stamp to unix time
unix_time = tritium_data['absolute_time'].astype('int64') // 10**9

counts, bin_edges = np.histogram(unix_time, bins = nbins)

# Compute the event rate
event_rate = counts / np.diff(bin_edges)

# average every 100 entries
average_length = 10
remainder_length = len(event_rate) % average_length
event_rate = np.mean(event_rate[:-remainder_length].reshape(-1,average_length),axis=1)

# convert unix time to time stamp
time_stamps = pd.to_datetime(bin_edges[:-remainder_length:average_length][:-1], unit='s')


#

fig, ax = plt.subplots(figsize=(15, 6))

ax.scatter(time_stamps, event_rate)

for date_time, comment in zip(df_important.date_time, df_important.comment):
    ax.axvline(date_time, color='r', linestyle="dashed")
    ax.text(date_time, 0.1, 
            f"{comment}", 
            rotation=90, 
            ha='left', 
            fontsize=12, 
            color='black')

# rotate x axis
plt.xticks(rotation=45)

plt.legend(bbox_to_anchor=(1.05, 0.5), loc='center left', fontsize=12)


##### Correlation with threshold?

In [ ]:
mask = (df_event_rate.absolute_time >= pd.Timestamp("2024-10-30 00:00:00")) & (
        util.vec_regex_search("10.0sig", df_event_rate.md_full_path)) & (
        # df_event_rate.integral_area_Vns_board_channel0 > 20) & (
        df_event_rate.voltage_preamp1_V == -46) & (
        # df_event_rate.board == 1) & (
        df_event_rate.integral_area_PE_board_channel0 > 2.5)

tritium_data = df_event_rate[mask]

number_of_events = tritium_data.groupby(['md_full_path'])['board'].count().to_numpy()
start_time = tritium_data.groupby(['md_full_path'])['absolute_time'].min()
end_time = tritium_data.groupby(['md_full_path'])['absolute_time'].max()
area = tritium_data.groupby(['md_full_path'])['integral_area_PE_board_channel0'].mean().to_numpy()
threshold_adc = tritium_data.groupby(['md_full_path'])['threshold_adc'].mean().to_numpy()
voltage_preamp1_V = tritium_data.groupby(['md_full_path'])['voltage_preamp1_V'].mean().to_numpy()
mean_baseline_std_V = tritium_data.groupby(['md_full_path'])['mean_baseline_std_V'].mean().to_numpy()


delta_time = (end_time - start_time).dt.total_seconds()
event_rate = number_of_events / delta_time

fig, ax = plt.subplots(figsize=(20, 4))
ax2 = ax.twinx()

for voltage in np.unique(voltage_preamp1_V):
    mask = voltage_preamp1_V == voltage
    ax.scatter(start_time[mask], event_rate[mask], s = area[mask], 
               label=f"Voltage: {voltage:.2f} V")

    ax2.scatter(start_time[mask], mean_baseline_std_V[mask], s = area[mask], 
               label=f"Voltage: {voltage:.2f} V", color = 'green')

for date_time, comment in zip(df_important.date_time, df_important.comment):
    ax.axvline(date_time, color='r', linestyle="dashed")
    ax.text(date_time, 0.1, 
            f"{comment}", 
            rotation=90, 
            ha='left', 
            fontsize=12, 
            color='black')

# rotate x axis
plt.xticks(rotation=45)

plt.legend(bbox_to_anchor=(1.05, 0.5), loc='center left', fontsize=12)


##### Correlation with threshold?


In [ ]:
mask = (df_event_rate.absolute_time >= pd.Timestamp("2024-10-30 00:00:00")) & (
        util.vec_regex_search("10.0sig", df_event_rate.md_full_path)) & (
        df_event_rate.integral_area_Vns_board_channel0 > 20) & (
        df_event_rate.voltage_preamp1_V == -46) & (
        df_event_rate.board == 0) & (
        df_event_rate.integral_area_Vns_board_channel0 > 0)

tritium_data = df_event_rate[mask]

number_of_events = tritium_data.groupby(['md_full_path'])['board'].count().to_numpy()
start_time = tritium_data.groupby(['md_full_path'])['absolute_time'].min()
end_time = tritium_data.groupby(['md_full_path'])['absolute_time'].max()
area = tritium_data.groupby(['md_full_path'])['integral_area_Vns_board_channel0'].mean().to_numpy()
threshold_adc = tritium_data.groupby(['md_full_path'])['threshold_adc'].mean().to_numpy()
voltage_preamp1_V = tritium_data.groupby(['md_full_path'])['voltage_preamp1_V'].mean().to_numpy()


delta_time = (end_time - start_time).dt.total_seconds()
event_rate = number_of_events / delta_time

fig, ax = plt.subplots(figsize=(20, 4))
ax2 = ax.twinx()

for voltage in np.unique(voltage_preamp1_V):
    mask = voltage_preamp1_V == voltage
    ax.scatter(start_time[mask], event_rate[mask], s = area[mask], 
               label=f"Voltage: {voltage:.2f} V")

    ax2.scatter(start_time[mask], threshold_adc[mask], s = area[mask], 
               label=f"Voltage: {voltage:.2f} V", color = 'green')

for date_time, comment in zip(df_important.date_time, df_important.comment):
    ax.axvline(date_time, color='r', linestyle="dashed")
    ax.text(date_time, 0.1, 
            f"{comment}", 
            rotation=90, 
            ha='left', 
            fontsize=12, 
            color='black')

# rotate x axis
plt.xticks(rotation=45)

plt.legend(bbox_to_anchor=(1.05, 0.5), loc='center left', fontsize=12)


In [ ]:
mask = (df_event_rate.absolute_time >= pd.Timestamp("2024-10-30 00:00:00")) & (
        # util.vec_regex_search("10.0sig", df_event_rate.md_full_path)) & (
        df_event_rate.integral_area_Vns_board_channel0 > 100) & (
        df_event_rate.voltage_preamp1_V == -46) & (
        # df_event_rate.board == 1) & (
        df_event_rate.integral_area_Vns_board_channel0 > 0)

tritium_data = df_event_rate[mask]

number_of_events = tritium_data.groupby(['md_full_path'])['board'].count().to_numpy()
start_time = tritium_data.groupby(['md_full_path'])['absolute_time'].min()
end_time = tritium_data.groupby(['md_full_path'])['absolute_time'].max()
area = tritium_data.groupby(['md_full_path'])['integral_area_Vns_board_channel0'].mean().to_numpy()
threshold_adc = tritium_data.groupby(['md_full_path'])['threshold_adc'].mean().to_numpy()
voltage_preamp1_V = tritium_data.groupby(['md_full_path'])['voltage_preamp1_V'].mean().to_numpy()


delta_time = (end_time - start_time).dt.total_seconds()
event_rate = number_of_events / delta_time

fig, ax = plt.subplots(figsize=(20, 4))
ax2 = ax.twinx()


ax.scatter(start_time, event_rate, c = area)


for date_time, comment in zip(df_important.date_time, df_important.comment):
    ax.axvline(date_time, color='r', linestyle="dashed")
    ax.text(date_time, 0.1, 
            f"{comment}", 
            rotation=90, 
            ha='left', 
            fontsize=12, 
            color='black')

# rotate x axis
plt.xticks(rotation=45)

# color bar
cbar = plt.colorbar(ax.collections[0], ax=ax, orientation='vertical')
cbar.set_label('Area', fontsize=12)

In [ ]:
mask = (df_event_rate.absolute_time >= pd.Timestamp("2024-10-30 00:00:00")) & (
        # util.vec_regex_search("10.0sig", df_event_rate.md_full_path)) & (
        df_event_rate.integral_area_Vns_board_channel0 > 10) & (
        df_event_rate.voltage_preamp1_V == -46) & (
        df_event_rate.board == 0)

tritium_data = df_event_rate[mask]

number_of_events = tritium_data.groupby(['md_full_path'])['board'].count().to_numpy()
start_time = tritium_data.groupby(['md_full_path'])['absolute_time'].min()
end_time = tritium_data.groupby(['md_full_path'])['absolute_time'].max()
area = tritium_data.groupby(['md_full_path'])['integral_area_Vns_board_channel0'].mean().to_numpy()
threshold_adc = tritium_data.groupby(['md_full_path'])['threshold_adc'].mean().to_numpy()
voltage_preamp1_V = tritium_data.groupby(['md_full_path'])['voltage_preamp1_V'].mean().to_numpy()


delta_time = (end_time - start_time).dt.total_seconds()
event_rate = number_of_events / delta_time

fig, ax = plt.subplots(figsize=(20, 7))

ax.scatter(start_time, event_rate, c = area)


for date_time, comment in zip(df_important.date_time, df_important.comment):
    ax.axvline(date_time, color='r', linestyle="dashed")
    ax.text(date_time, 180, 
            f"{comment}", 
            rotation=45, 
            ha='left', 
        #     fontsize=12, 
            color='black')
    
    

# rotate x axis
plt.xticks(rotation=30)

#format date time axis
ax.xaxis.set_major_formatter(plt.matplotlib.dates.DateFormatter('%Y-%m-%d %H:%M'))

# color bar
cbar = plt.colorbar(ax.collections[0], ax=ax, orientation='vertical')
cbar.set_label('Integral Area [PE]')

ax.set_xlabel('Time')
ax.set_ylabel('Event Rate [Hz]')

# plt.savefig('/kalinka/storage/darkmatter/XENONnT/sk6801/UCSD_data/plots/peak_all_channel_event_rate.pdf', dpi=100, bbox_inches='tight')


In [ ]:
mask = (df_event_rate.absolute_time >= pd.Timestamp("2024-10-30 00:00:00")) & (
        # util.vec_regex_search("10.0sig", df_event_rate.md_full_path)) & (
        df_event_rate.integral_area_PE_board_channel0 > 20) & (
        df_event_rate.voltage_preamp1_V == -46) & (
        df_event_rate.board == 0)

tritium_data = df_event_rate[mask]

number_of_events = tritium_data.groupby(['md_full_path'])['board'].count().to_numpy()
start_time = tritium_data.groupby(['md_full_path'])['absolute_time'].min()
end_time = tritium_data.groupby(['md_full_path'])['absolute_time'].max()
area = tritium_data.groupby(['md_full_path'])['integral_area_Vns_board_channel0'].mean().to_numpy()
threshold_adc = tritium_data.groupby(['md_full_path'])['threshold_adc'].mean().to_numpy()
voltage_preamp1_V = tritium_data.groupby(['md_full_path'])['voltage_preamp1_V'].mean().to_numpy()


delta_time = (end_time - start_time).dt.total_seconds()
event_rate = number_of_events / delta_time

fig, ax = plt.subplots(figsize=(20, 7))

ax.scatter(start_time, event_rate, c = area)


for date_time, comment in zip(df_important.date_time, df_important.comment):
    ax.axvline(date_time, color='r', linestyle="dashed")
    ax.text(date_time, 180, 
            f"{comment}", 
            rotation=45, 
            ha='left', 
        #     fontsize=12, 
            color='black')
    
    

# rotate x axis
plt.xticks(rotation=30)

#format date time axis
ax.xaxis.set_major_formatter(plt.matplotlib.dates.DateFormatter('%Y-%m-%d %H:%M'))

# color bar
cbar = plt.colorbar(ax.collections[0], ax=ax, orientation='vertical')
cbar.set_label('Integral Area [PE]')

ax.set_xlabel('Time')
ax.set_ylabel('Event Rate [Hz]')

# plt.savefig('/kalinka/storage/darkmatter/XENONnT/sk6801/UCSD_data/plots/peak_all_channel_event_rate.pdf', dpi=100, bbox_inches='tight')


In [ ]:
fig, ax = plt.subplots(1,4, figsize=(20, 7), sharey=True)


run_tag_list = ["LXe/gain_calibration", "LXe/Cs137", "LXe/Co57", "LXe/tritium"]

for i, run_tag in enumerate(run_tag_list):
        mask = (df_event_rate.run_tag == run_tag) & (
                # util.vec_regex_search("10.0sig", df_event_rate.md_full_path)) & (
                df_event_rate.integral_area_PE_board_channel0 > 10) & (
                (df_event_rate.voltage_preamp1_V <= -46))

        data = df_event_rate[mask]

        number_of_events = data.groupby(['md_full_path'])['board'].count().to_numpy()
        start_time = data.groupby(['md_full_path'])['absolute_time'].min()
        end_time = data.groupby(['md_full_path'])['absolute_time'].max()
        area = data.groupby(['md_full_path'])['integral_area_Vns_board_channel0'].mean().to_numpy()
        threshold_adc = data.groupby(['md_full_path'])['threshold_adc'].mean().to_numpy()
        voltage_preamp1_V = data.groupby(['md_full_path'])['voltage_preamp1_V'].mean().to_numpy()


        delta_time = (end_time - start_time).dt.total_seconds()
        event_rate = number_of_events / delta_time

        ax[i].scatter(start_time, event_rate, c = area)
        ax[i].set_title(run_tag, fontsize=16)
        #format date time axis
        ax[i].xaxis.set_major_formatter(plt.matplotlib.dates.DateFormatter('%Y-%m-%d %H:%M'))



# for date_time, comment in zip(df_important.date_time, df_important.comment):
#     ax[3].axvline(date_time, color='r', linestyle="dashed")
#     ax[3].text(date_time, 180, 
#             f"{comment}", 
#             rotation=45, 
#             ha='left', 
#         #     fontsize=12, 
#             color='black')
    
    

# rotate x axis
plt.xticks(rotation=30)

# color bar
cbar = plt.colorbar(ax[3].collections[0], ax=ax[3], orientation='vertical')
cbar.set_label('Integral Area [PE]')

plt.xlabel('Time')
plt.ylabel('Event Rate [Hz]')

# plt.savefig('/kalinka/storage/darkmatter/XENONnT/sk6801/UCSD_data/plots/peak_all_channel_event_rate.pdf', dpi=100, bbox_inches='tight')


In [ ]:
mask = (df_event_rate.absolute_time >= pd.Timestamp("2024-10-30 00:00:00")) & (
        # util.vec_regex_search("10.0sig", df_event_rate.md_full_path)) & (
        df_event_rate.integral_area_Vns_board_channel0 > 30) & (
        df_event_rate.integral_area_Vns_board_channel0 < 40) & (
        df_event_rate.voltage_preamp1_V == -46) & (
        df_event_rate.board == 0)

tritium_data = df_event_rate[mask]

number_of_events = tritium_data.groupby(['md_full_path'])['board'].count().to_numpy()
start_time = tritium_data.groupby(['md_full_path'])['absolute_time'].min()
end_time = tritium_data.groupby(['md_full_path'])['absolute_time'].max()
area = tritium_data.groupby(['md_full_path'])['integral_area_Vns_board_channel0'].mean().to_numpy()
threshold_adc = tritium_data.groupby(['md_full_path'])['threshold_adc'].mean().to_numpy()
voltage_preamp1_V = tritium_data.groupby(['md_full_path'])['voltage_preamp1_V'].mean().to_numpy()


delta_time = (end_time - start_time).dt.total_seconds()
event_rate = number_of_events / delta_time

fig, ax = plt.subplots(figsize=(20, 7))

ax.scatter(start_time, event_rate, c = area)


for date_time, comment in zip(df_important.date_time, df_important.comment):
    ax.axvline(date_time, color='r', linestyle="dashed")
    ax.text(date_time, 21, 
            f"{comment}", 
            rotation=45, 
            ha='left', 
        #     fontsize=12, 
            color='black')
    
    

# rotate x axis
plt.xticks(rotation=30)

#format date time axis
ax.xaxis.set_major_formatter(plt.matplotlib.dates.DateFormatter('%Y-%m-%d %H:%M'))

# color bar
cbar = plt.colorbar(ax.collections[0], ax=ax, orientation='vertical')
cbar.set_label('Integral Area [PE]')

ax.set_xlabel('Time')
ax.set_ylabel('Event Rate [Hz]')

# plt.savefig('/kalinka/storage/darkmatter/XENONnT/sk6801/UCSD_data/plots/peak_all_channel_event_rate.pdf', dpi=100, bbox_inches='tight')


##### At different threshold multiplier
but a cut on area would remove this effect

In [ ]:
fig, ax = plt.subplots(figsize=(20, 4))

for threshold_sig in np.arange(5, 11):
        mask = util.vec_regex_search(f"{int(threshold_sig)}.0sig", df_event_rate.md_full_path) & (
        df_event_rate.absolute_time >= pd.Timestamp("2024-10-30 00:00:00")) & (
        # util.vec_regex_search("10.0sig", df_event_rate.md_full_path)) & (
        df_event_rate.integral_area_Vns_board_channel0 > 0) & (
        df_event_rate.voltage_preamp1_V == -46) & (
        # df_event_rate.board == 1) & (
        df_event_rate.integral_area_Vns_board_channel0 > 0)

        tritium_data = df_event_rate[mask]

        number_of_events = tritium_data.groupby(['md_full_path'])['board'].count().to_numpy()
        start_time = tritium_data.groupby(['md_full_path'])['absolute_time'].min()
        end_time = tritium_data.groupby(['md_full_path'])['absolute_time'].max()
        area = tritium_data.groupby(['md_full_path'])['integral_area_Vns_board_channel0'].mean().to_numpy()
        threshold_adc = tritium_data.groupby(['md_full_path'])['threshold_adc'].mean().to_numpy()
        voltage_preamp1_V = tritium_data.groupby(['md_full_path'])['voltage_preamp1_V'].mean().to_numpy()

        delta_time = (end_time - start_time).dt.total_seconds()
        event_rate = number_of_events / delta_time

        ax.scatter(start_time, event_rate, s=area, label = f"{threshold_sig} sig")

for date_time, comment in zip(df_important.date_time, df_important.comment):
        ax.axvline(date_time, color='r', linestyle="dashed")
        ax.text(date_time, 0.1, 
                f"{comment}", 
                rotation=90, 
                ha='left', 
                fontsize=12, 
                color='black')

# rotate x axis
plt.xticks(rotation=45)

# color bar
# cbar = plt.colorbar(ax.collections[0], ax=ax, orientation='vertical')
# cbar.set_label('Area', fontsize=12)

plt.legend(bbox_to_anchor=(1.05, 0.5), loc='center left', fontsize=12)

##### For all data

In [ ]:
mask = (
    # df_event_rate.absolute_time >= pd.Timestamp("2024-10-30 00:00:00")) & (
        # util.vec_regex_search("10.0sig", df_event_rate.md_full_path)) & (
        # df_event_rate.voltage_preamp1_V == -46) & (
        # df_event_rate.board == 1) & (
        df_event_rate.integral_area_Vns_board_channel0 > 0)

tritium_data = df_event_rate[mask]

number_of_events = tritium_data.groupby(['md_full_path'])['board'].count().to_numpy()
start_time = tritium_data.groupby(['md_full_path'])['absolute_time'].min()
end_time = tritium_data.groupby(['md_full_path'])['absolute_time'].max()
area = tritium_data.groupby(['md_full_path'])['integral_area_Vns_board_channel0'].mean().to_numpy()
threshold_adc = tritium_data.groupby(['md_full_path'])['threshold_adc'].mean().to_numpy()
voltage_preamp1_V = tritium_data.groupby(['md_full_path'])['voltage_preamp1_V'].mean().to_numpy()


delta_time = (end_time - start_time).dt.total_seconds()
event_rate = number_of_events / delta_time

fig, ax = plt.subplots(figsize=(20, 4))

for voltage in np.unique(voltage_preamp1_V):
    mask = voltage_preamp1_V == voltage
    ax.scatter(start_time[mask], event_rate[mask], s = area[mask], 
               label=f"Voltage: {voltage:.2f} V")
    

for date_time, comment in zip(df_important.date_time, df_important.comment):
    ax.axvline(date_time, color='r', linestyle="dashed")
    ax.text(date_time, 0.1, 
            f"{comment}", 
            rotation=90, 
            ha='left', 
            fontsize=12, 
            color='black')

# rotate x axis
plt.xticks(rotation=45)

plt.legend(bbox_to_anchor=(1.05, 0.5), loc='center left', fontsize=12)


#### Average every 100 seconds

In [ ]:
tritium_time = df_event_rate.absolute_time[df_event_rate.absolute_time >= pd.Timestamp("2024-10-30")]

event_max = np.max(tritium_time)
event_min = np.min(tritium_time)
n_bins = int((event_max - event_min).total_seconds()) # 1 second per bin


mask = df_event_rate.absolute_time >= pd.Timestamp("2024-10-30")
#convert pd timestamp to unix time
unix_time = df_event_rate['absolute_time'][mask].astype(np.int64) // 10**9

event_rate, event_time = np.histogram(unix_time, bins = n_bins)

# convert unix time to pd timestamp
event_time = pd.to_datetime(event_time, unit='s')

# average event rate for every 100 entries

# remove last few entries to make it divisible by 100
tmp = len(event_rate)%100
event_rate = event_rate[:-tmp]

n_bins_after_average = len(event_rate) // 100

event_rate = event_rate.reshape(-1, 100).mean(axis=1)
event_time = event_time[::100][:-1]

In [ ]:
fig, ax = plt.subplots(figsize=(20, 4))


ax.scatter(event_time, event_rate)
for date_time, comment in zip(df_important.date_time, df_important.comment):
    ax.axvline(date_time, color='r', linestyle="dashed")
    ax.text(date_time, 0.1, 
            f"{comment}", 
            rotation=90, 
            ha='left', 
            fontsize=12, 
            color='black')

# rotate x axis
plt.xticks(rotation=45)
